In [1]:
import pandas as pd
import numpy as np
import re
import torch
import scipy.stats as stats

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.cluster import AgglomerativeClustering
from sklearn.base import clone
from nltk.stem import PorterStemmer

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

# Working with Text Lab
## Information retrieval, preprocessing, and feature extraction

In this lab, you'll be looking at and exploring European restaurant reviews. The dataset is rather tiny, but that's just because it has to run on any machine. In real life, just like with images, texts can be several terabytes long.

The dataset is located [here](https://www.kaggle.com/datasets/gorororororo23/european-restaurant-reviews) and as always, it's been provided to you in the `data/` folder.

### Problem 1. Read the dataset (1 point)
Read the dataset, get acquainted with it. Ensure the data is valid before you proceed.

How many observations are there? Which country is the most represented? What time range does the dataset represent?

Is the sample balanced in terms of restaurants, i.e., do you have an equal number of reviews for each one? Most importantly, is the dataset balanced in terms of **sentiment**?

In [2]:
reviews = pd.read_csv("data/European Restaurant Reviews.csv")
reviews

,Country,Restaurant Name,Sentiment,Review Title,Review Date,Review
0,France,The Frog at Bercy Village,Negative,Rude manager,May 2024 •,The manager became agressive when I said the c...
1,France,The Frog at Bercy Village,Negative,A big disappointment,Feb 2024 •,"I ordered a beef fillet ask to be done medium,..."
2,France,The Frog at Bercy Village,Negative,Pretty Place with Bland Food,Nov 2023 •,"This is an attractive venue with welcoming, al..."
3,France,The Frog at Bercy Village,Negative,Great service and wine but inedible food,Mar 2023 •,Sadly I used the high TripAdvisor rating too ...
4,France,The Frog at Bercy Village,Negative,Avoid- Worst meal in Rome - possibly ever,Nov 2022 •,From the start this meal was bad- especially g...
...,...,...,...,...,...,...
1497,Cuba,Old Square (Plaza Vieja),Negative,The Tourism Trap,Oct 2016 •,Despite the other reviews saying that this is ...
1498,Cuba,Old Square (Plaza Vieja),Negative,the beer factory,Oct 2016 •,beer is good. food is awfull The only decent...
1499,Cuba,Old Square (Plaza Vieja),Negative,brewery,Oct 2016 •,"for terrible service of a truly comedic level,..."
1500,Cuba,Old Square (Plaza Vieja),Negative,It's nothing exciting over there,Oct 2016 •,We visited the Havana's Club Museum which is l...


In [3]:
reviews.dtypes

Country            object
Restaurant Name    object
Sentiment          object
Review Title       object
Review Date        object
Review             object
dtype: object

We clean Review Date column: replace 'Sept' with 'Sep', remove '•'. Then, convert to datetime (parse Month Year).

In [4]:
reviews['Review Date'] = reviews['Review Date'].str.replace('Sept', 'Sep').str.replace('•','').str.strip()

In [5]:
reviews['Review Date'] = pd.to_datetime(reviews['Review Date'], format='%b %Y')

In [6]:
reviews

,Country,Restaurant Name,Sentiment,Review Title,Review Date,Review
0,France,The Frog at Bercy Village,Negative,Rude manager,2024-05-01,The manager became agressive when I said the c...
1,France,The Frog at Bercy Village,Negative,A big disappointment,2024-02-01,"I ordered a beef fillet ask to be done medium,..."
2,France,The Frog at Bercy Village,Negative,Pretty Place with Bland Food,2023-11-01,"This is an attractive venue with welcoming, al..."
3,France,The Frog at Bercy Village,Negative,Great service and wine but inedible food,2023-03-01,Sadly I used the high TripAdvisor rating too ...
4,France,The Frog at Bercy Village,Negative,Avoid- Worst meal in Rome - possibly ever,2022-11-01,From the start this meal was bad- especially g...
...,...,...,...,...,...,...
1497,Cuba,Old Square (Plaza Vieja),Negative,The Tourism Trap,2016-10-01,Despite the other reviews saying that this is ...
1498,Cuba,Old Square (Plaza Vieja),Negative,the beer factory,2016-10-01,beer is good. food is awfull The only decent...
1499,Cuba,Old Square (Plaza Vieja),Negative,brewery,2016-10-01,"for terrible service of a truly comedic level,..."
1500,Cuba,Old Square (Plaza Vieja),Negative,It's nothing exciting over there,2016-10-01,We visited the Havana's Club Museum which is l...


In [7]:
reviews.dtypes

Country                    object
Restaurant Name            object
Sentiment                  object
Review Title               object
Review Date        datetime64[ns]
Review                     object
dtype: object

In [8]:
num_observations = len(reviews)

In [9]:
most_country = reviews['Country'].value_counts().idxmax()

In [10]:
date_min = reviews['Review Date'].min()
date_max = reviews['Review Date'].max()

In [11]:
restaurant_counts = reviews['Restaurant Name'].value_counts()

In [12]:
sentiment_counts = reviews['Sentiment'].value_counts()

In [13]:
print(f"Total reviews: {num_observations}")
print(f"Most reviews from: {most_country}")
print(f"Date range: {date_min.strftime('%b %Y')} to {date_max.strftime('%b %Y')}\n")

Total reviews: 1502
Most reviews from: France
Date range: Sep 2010 to Jul 2024



In [14]:
print("Reviews per restaurant:")
for name, count in restaurant_counts.items():
    print(f"  {name}: {count}")

Reviews per restaurant:
  The Frog at Bercy Village: 512
  Ad Hoc Ristorante (Piazza del Popolo): 318
  The LOFT: 210
  Old Square (Plaza Vieja): 146
  Stara Kamienica: 135
  Pelmenya: 100
  Mosaic: 81


For restaurant balance: no, not at all - review counts range from 81 (Mosaic) up to 512 (The Frog at Bercy Village), so some restaurants are heavily over‑represented while others have relatively few reviews.

In [15]:
print("\nSentiment distribution:")
for sentiment, count in sentiment_counts.items():
    print(f"  {sentiment}: {count}")


Sentiment distribution:
  Positive: 1237
  Negative: 265


For sentiment balance: also not balanced at all - there are 1237 positive vs only 265 negative reviews, so roughly 80% positive and 20% negative.

### Problem 2. Getting acquainted with reviews (1 point)
Are positive comments typically shorter or longer? Try to define a good, robust metric for "length" of a text; it's not necessary just the character count. Can you explain your findings?

In [16]:
# We define length metrics
reviews['word_count'] = reviews['Review'].str.split().apply(len)

In [17]:
length_stats = reviews.groupby('Sentiment')['word_count'] \
                 .agg(['mean', 'median', 'std', 'count']) \
                 .reset_index()
length_stats

,Sentiment,mean,median,std,count
0,Negative,140.573585,95.0,131.759636,265
1,Positive,50.183508,37.0,38.741043,1237


Here are our observations: 

Negative reviews (265 total):

Mean: $\sim$ 140 words

Median: 95 words

Std Dev: $\sim$ 132 words

Positive reviews (1237 total):

Mean: $\sim$  50 words

Median: 37 words

Std Dev: $\sim$ 39 words

I used word count (number of words per review) as a robust length metric rather than simple character count, since it better reflects how much content a review contains.

My findings are connected with the fact that negative comments are substantially longer than positive ones (mean of 140 vs. 50 words). This makes sense because complaints typically require more explanation, examples and detail, whereas positive feedback is often briefer ("Great service!", "Loved it!"). The higher standard deviation for negatives also shows that some negative reviews get quite wordy. This demonstrates that sentiment correlates with review length: longer reviews tend to be negative in this dataset. 

### Problem 3. Preprocess the review content (2 points)
You'll likely need to do this while working on the problems below, but try to synthesize (and document!) your preprocessing here. Your tasks will revolve around words and their connection to sentiment. While preprocessing, keep in mind the domain (restaurant reviews) and the task (sentiment analysis).

We start with: 

1. Text Normalization - lowercasing: we need to convert all characters to lowercase to reduce vocabulary size (Great -> great). Next, we remove HTML artifacts and special symbols: strip tags and uncommon unicode (&amp;, emojis) that don't contribute to sentiment.

In [18]:
def normalize_text(text):
    """
    Function to normalize a given text - remove artifacts, symbols, convert to lowercase
    """
    
    text = text.lower()
    text = re.sub(r"&amp;|&lt;|&gt;", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)    # remove punctuation and special chars
    return text

In [19]:
reviews['CleanReview'] = reviews['Review'].apply(normalize_text)
reviews['CleanReview']

0       the manager became agressive when i said the c...
1       i ordered a beef fillet ask to be done medium ...
2       this is an attractive venue with welcoming  al...
3       sadly i  used the high tripadvisor rating too ...
4       from the start this meal was bad  especially g...
                              ...                        
1497    despite the other reviews saying that this is ...
1498    beer is good   food is awfull  the only decent...
1499    for terrible service of a truly comedic level ...
1500    we visited the havana s club museum which is l...
1501    food and service was awful  very pretty stop  ...
Name: CleanReview, Length: 1502, dtype: object

2. Tokenization - whitespace tokenization - the simplest and fastest approach is to split on whitespace. This works well for our cleaned text.

In [20]:
reviews['Tokens'] = reviews['CleanReview'].str.split()
reviews['Tokens']

0       [the, manager, became, agressive, when, i, sai...
1       [i, ordered, a, beef, fillet, ask, to, be, don...
2       [this, is, an, attractive, venue, with, welcom...
3       [sadly, i, used, the, high, tripadvisor, ratin...
4       [from, the, start, this, meal, was, bad, espec...
                              ...                        
1497    [despite, the, other, reviews, saying, that, t...
1498    [beer, is, good, food, is, awfull, the, only, ...
1499    [for, terrible, service, of, a, truly, comedic...
1500    [we, visited, the, havana, s, club, museum, wh...
1501    [food, and, service, was, awful, very, pretty,...
Name: Tokens, Length: 1502, dtype: object

3. Stopword Removal - instead of manual lists, we will use scikit-learn's built-in removal in one step during feature extraction.

In [21]:
vectorizer = TfidfVectorizer(
    stop_words='english',  
    tokenizer= lambda text: text.split(),
    lowercase=False,
    token_pattern = None
)

X_tfidf = vectorizer.fit_transform(reviews['CleanReview'])
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 42447 stored elements and shape (1502, 6032)>

4. Stemming - to avoid external model dependencies, we will apply Porter stemming using NLTK's built-in stemmer.

In [22]:
stemmer = PorterStemmer()

In [23]:
def stem_tokens(tokens):
    """
    Function to stem received tokens
    """
    
    return [stemmer.stem(t) for t in tokens]

In [24]:
reviews['Stems'] = reviews['Tokens'].apply(stem_tokens)
reviews['Stems']

0       [the, manag, becam, agress, when, i, said, the...
1       [i, order, a, beef, fillet, ask, to, be, done,...
2       [thi, is, an, attract, venu, with, welcom, alb...
3       [sadli, i, use, the, high, tripadvisor, rate, ...
4       [from, the, start, thi, meal, wa, bad, especi,...
                              ...                        
1497    [despit, the, other, review, say, that, thi, i...
1498    [beer, is, good, food, is, awful, the, onli, d...
1499    [for, terribl, servic, of, a, truli, comed, le...
1500    [we, visit, the, havana, s, club, museum, whic...
1501    [food, and, servic, wa, aw, veri, pretti, stop...
Name: Stems, Length: 1502, dtype: object

5. Handling Negation - we will apply only if we need to capture flipped sentiment ("not good").

In [25]:
def handle_negation(tokens):
    """
    Function to handle the negation of words with received tokens
    """
    
    out = []
    neg = False
    for t in tokens:
        if t in {'not', 'no', 'never'}:
            neg = True
            continue
        if neg:
            out.append(f"not_{t}")
            neg = False
        else:
            out.append(t)
    return out

In [26]:
reviews['NegTokens'] = reviews['Stems'].apply(handle_negation)
reviews['NegTokens']

0       [the, manag, becam, agress, when, i, said, the...
1       [i, order, a, beef, fillet, ask, to, be, done,...
2       [thi, is, an, attract, venu, with, welcom, alb...
3       [sadli, i, use, the, high, tripadvisor, rate, ...
4       [from, the, start, thi, meal, wa, bad, especi,...
                              ...                        
1497    [despit, the, other, review, say, that, thi, i...
1498    [beer, is, good, food, is, awful, the, onli, d...
1499    [for, terribl, servic, of, a, truli, comed, le...
1500    [we, visit, the, havana, s, club, museum, whic...
1501    [food, and, servic, wa, aw, veri, pretti, stop...
Name: NegTokens, Length: 1502, dtype: object

### Problem 4. Top words (1 point)
Use a simple word tokenization and count the top 10 words in positive reviews; then the top 10 words in negative reviews*. Once again, try to define what "top" words means. Describe and document your process. Explain your results.

\* Okay, you may want to see top N words (with $N \ge 10$).

We define "top" words as those with the highest term frequency (raw count) within each sentiment subset.
We exclude any tokens of length $\le$ 2 (like "it", "in") to focus on more meaningful words.

In [27]:
# Build stopword sets
tfidf_stop = set(vectorizer.get_stop_words())
domain_stop = {'food', 'service', 'restaurant', 'menu', 'place', 'wine', 'staff'}

In [28]:
def filter_tokens(tok_list):
    """
    Function to filter out short tokens, TF-IDF stopwords, and domain words
    """
    
    return [
        t for t in tok_list
        if len(t) > 2
        and t not in tfidf_stop
        and t not in domain_stop
    ]

In [29]:
# Gather tokens by sentiment
positive_tokens = reviews.loc[reviews['Sentiment'] == 'Positive', 'Tokens'].sum()
negative_tokens = reviews.loc[reviews['Sentiment'] == 'Negative', 'Tokens'].sum()

positive_filtered = filter_tokens(positive_tokens)
negative_filtered = filter_tokens(negative_tokens)

# Count frequencies
positive_counts = Counter(positive_filtered)
negative_counts = Counter(negative_filtered)

# Compute difference and get top N
diff = {}
all_terms = set(positive_counts) | set(negative_counts)
for term in all_terms:
    diff[term] = positive_counts.get(term, 0) - negative_counts.get(term, 0)

top_N = 10

# Top N distinctly positive (largest positive diff)
top_positives = sorted(diff.items(), key=lambda x: -x[1])[:top_N]

# Top N distinctly negative (most negative diff)
top_negatives = sorted(diff.items(), key=lambda x: x[1])[:top_N]

print("Top 10 Distinctly Positive Words:")
for w, d in top_positives:
    print(f"  {w:15} (+{d})")

print("\nTop 10 Distinctly Negative Words:")
for w, d in top_negatives:
    print(f"  {w:15} ({d})")

Top 10 Distinctly Positive Words:
  great           (+520)
  good            (+361)
  excellent       (+221)
  delicious       (+221)
  nice            (+220)
  friendly        (+218)
  amazing         (+189)
  recommend       (+171)
  atmosphere      (+161)
  lovely          (+154)

Top 10 Distinctly Negative Words:
  table           (-77)
  asked           (-75)
  minutes         (-61)
  average         (-53)
  told            (-46)
  cold            (-46)
  said            (-41)
  reviews         (-40)
  took            (-38)
  did             (-37)


These "distinctly" lists look solid. They show the words most over‑represented in each class after filtering out generic terms. Distinctly Positive: great, good, excellent, delicious, etc., are exactly the praise words we'd expect, each with a large positive difference (like 520 more occurrences in positives than negatives). Distinctly Negative: table, asked, minutes, average, cold, etc., reflect typical complaints about waits, seating, or subpar food.

### Problem 5. Review titles (2 point)
How do the top words you found in the last problem correlate to the review titles? Do the top 10 words (for each sentiment) appear in the titles at all? Do reviews which contain one or more of the top words have the same words in their titles?

Does the title of a comment present a good summary of its content? That is, are the titles descriptive, or are they simply meant to catch the attention of the reader?

We normalize and tokenize both review bodies and titles in the same way as the main text. For each previously identified top word (separately for positive and negative sentiment), we measure three things: how many reviews contain the word in their body, how many titles contain that word at all, and of those reviews that contain the word in the body, the fraction whose title also contains it. This quantifies both the presence of sentiment-bearing vocabulary in titles and the degree to which titles explicitly reuse the same words from the review content.

In [30]:
reviews['CleanTitle'] = reviews['Review Title'].apply(normalize_text)
reviews['CleanTitle'] 

0                                    rude manager
1                            a big disappointment
2                    pretty place with bland food
3        great service and wine but inedible food
4       avoid  worst meal in rome   possibly ever
                          ...                    
1497                             the tourism trap
1498                             the beer factory
1499                                      brewery
1500             it s nothing exciting over there
1501                                 tourist trap
Name: CleanTitle, Length: 1502, dtype: object

In [31]:
reviews['TitleTokens'] = reviews['CleanTitle'].str.split()
reviews['TitleTokens']

0                                        [rude, manager]
1                               [a, big, disappointment]
2                     [pretty, place, with, bland, food]
3       [great, service, and, wine, but, inedible, food]
4         [avoid, worst, meal, in, rome, possibly, ever]
                              ...                       
1497                                [the, tourism, trap]
1498                                [the, beer, factory]
1499                                           [brewery]
1500             [it, s, nothing, exciting, over, there]
1501                                     [tourist, trap]
Name: TitleTokens, Length: 1502, dtype: object

In [32]:
top_positive_words = [w for w, _ in top_positives]
top_negative_words = [w for w, _ in top_negatives]

We compute the proportion of reviews that contain at least one top sentiment word in their body and also have any of those words in their title. This provides a coarser summary of whether titles tend to echo the key sentiment words of their corresponding reviews.

In [33]:
def compute_word_title_overlap(top_words, sentiment_label):
    """
    Function to compute a word overlap in terms of a review title
    """
    
    subset = reviews[reviews['Sentiment'] == sentiment_label]
    records = []
    for w in top_words:
        in_review = subset['Tokens'].apply(lambda toks: w in toks)
        in_title = subset['TitleTokens'].apply(lambda toks: w in toks)

        reviews_with = in_review.sum()
        titles_with = in_title.sum()
        reuse_frac = ((in_review & in_title).sum()) / reviews_with if reviews_with > 0 else np.nan

        records.append({
            'word': w,
            'sentiment': sentiment_label,
            'reviews_with_word': int(reviews_with),
            'titles_with_word': int(titles_with),
            'fraction_title_reuses_word': reuse_frac
        })
        
    return pd.DataFrame(records)

To assess whether titles summarize review content or merely grab attention, we compute set-based similarity metrics between filtered title tokens and filtered review tokens. The Jaccard similarity measures the intersection-over-union of their token sets, giving a normalized overlap score

In [34]:
pos_overlap_df = compute_word_title_overlap(top_positive_words, 'Positive')
neg_overlap_df = compute_word_title_overlap(top_negative_words, 'Negative')

print("Positive top-word overlap with titles:")
print(pos_overlap_df.to_string(index=False))
print("\nNegative top-word overlap with titles:")
print(neg_overlap_df.to_string(index=False))

Positive top-word overlap with titles:
      word sentiment  reviews_with_word  titles_with_word  fraction_title_reuses_word
     great  Positive                432               210                    0.291667
      good  Positive                383                99                    0.164491
 excellent  Positive                207                77                    0.173913
 delicious  Positive                213                43                    0.140845
      nice  Positive                260                48                    0.088462
  friendly  Positive                231                26                    0.064935
   amazing  Positive                171                66                    0.175439
 recommend  Positive                187                12                    0.021390
atmosphere  Positive                181                32                    0.060773
    lovely  Positive                148                37                    0.114865

Negative top-w

Titles sometimes capture prominent positive sentiment but less so negative; overall overlap is limited, indicating titles lean toward highlighting or framing rather than verbatim summarization

We measured, for each of the top 10 "distinctly" positive and negative words, (a) how often they appear in review bodies, (b) how often they appear in titles, and (c) of reviews that contain the word, the fraction whose title also contains it. Positive sentiment words like "great" and "good" are somewhat echoed in titles ("great" is reused in roughly 30% of reviews that contain it), whereas most other positive words are reused far less frequently. On the negative side, almost none of the distinctive negative words (like "cold", "told", "said") are mirrored in titles - reuse fractions are essentially zero except for "average" (22%). This shows a weak-to-moderate correlation for a few strong positive terms and very little direct reuse for negative terms.

As for the question - "Do the top 10 words (for each sentiment) appear in the titles at all?" - yes for positives, rarely for negatives. Several positive top words appear in many titles, showing that positive reviewers sometimes front-load sentiment vocabulary in the title. In contrast, most top negative words scarcely appear in titles, so negative sentiment is usually not expressed with those same words in titles.

The highest reuse is for "great" (29% of such positive reviews), but for most other top words (both positive and negative), the fraction is well below 20%, often near zero. That means titles seldom verbatim reuse the body's most distinctive sentiment words; they may instead paraphrase, generalize, or choose different hooks.

The titles are partially descriptive but in many cases do not faithfully summarize the review content; instead, they often serve more as attention-grabbers or high-level tone indicators. This conclusion comes from measuring vocabulary overlap between title and body using set similarity (Jaccard) and recall (ROUGE-1). While some titles reuse key sentiment words and show high overlap - indicating they succinctly reflect the review - many have low overlap, especially when negative sentiment is expressed. Thus, titles sometimes hint at the sentiment but rarely provide a full or literal summary of the review's content. ROUGE-1 simply is: Of the words that appear in the title, what fraction also appear somewhere in the review. It asks: "Is the title grounded in the review?" Titles that use words not in the review get lower scores. For Jaccard: How much the title and the review share in vocabulary, relative to all the distinct words they use between them.

In [35]:
# Filter tokens
reviews['FilteredReviewTokens'] = reviews['Tokens'].apply(
    lambda toks: [t for t in toks if len(t) > 2 and t not in tfidf_stop]
)
reviews['FilteredTitleTokens'] = reviews['TitleTokens'].apply(
    lambda toks: [t for t in toks if len(t) > 2]
)

# Similarity metrics
def jaccard(a, b):
    """
    Function to evaluate Jaccard metric
    """
    
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def rouge1_recall(title_tokens, review_tokens):
    """
    Function to evaluate ROUGE-1 metric
    """
    
    ts = set(title_tokens)
    if not ts:
        return 0.0
    return len(ts & set(review_tokens)) / len(ts)

reviews['Jaccard'] = reviews.apply(
    lambda r: jaccard(r['FilteredTitleTokens'], r['FilteredReviewTokens']), axis=1
)

reviews['ROUGE1'] = reviews.apply(
    lambda r: rouge1_recall(r['TitleTokens'], r['Tokens']), axis=1
)

reviews['TitleLen'] = reviews['TitleTokens'].str.len()
reviews['ReviewLen'] = reviews['Tokens'].str.len()
reviews['LenRatio'] = reviews['TitleLen'] / reviews['ReviewLen'].replace(0, np.nan)

print("High overlap examples:\n", reviews.nlargest(3, 'Jaccard')[['Review Title', 'Review', 'Jaccard', 'ROUGE1']])
print("Low overlap examples:\n", reviews.nsmallest(3, 'Jaccard')[['Review Title', 'Review', 'Jaccard', 'ROUGE1']])

High overlap examples:
                                           Review Title  \
927  Wonderful restaurant! Best place for dinner. V...   
949  Beautiful kitchen and personal!!! I’m very hap...   
908  Delicious cuisine, great service and beautiful...   

                                                Review   Jaccard  ROUGE1  
927  Wonderful restaurant! Best place for dinner. V...  0.857143     1.0  
949  Beautiful kitchen and personal!!! I’m very hap...  0.666667     1.0  
908  Delicious cuisine, great service and beautiful...  0.500000     1.0  
Low overlap examples:
                           Review Title  \
1                 A big disappointment   
7                  Huge Disappointment   
11  Perfectly organized absolute scam!   

                                               Review  Jaccard    ROUGE1  
1   I ordered a beef fillet ask to be done medium,...      0.0  0.333333  
7   This restaurant’s high rating is wholly unwarr...      0.0  0.000000  
11  The place offers reall

To conclude, titles only partially summarize their reviews. Strong positive words (like "great") show up in titles somewhat often, but most distinctive words - especially negative ones - are rarely reused verbatim, so overlap is low. Quantitatively, modest Jaccard and ROUGE-1 scores confirm that many titles diverge from the body and act more as hooks or tone markers than full summaries. For downstream work, the title can be a useful auxiliary signal (especially for positive sentiment) but shouldn't be trusted alone as a faithful summary.

### Problem 6. Bag of words (1 point)
Based on your findings so far, come up with a good set of settings (hyperparameters) for a bag-of-words model for review titles and contents. It's easiest to treat them separately (so, create two models); but you may also think about a unified representation. I find the simplest way of concatenating the title and content too simplistic to be useful, as it doesn't allow you to treat the title differently (e.g., by giving it more weight).

The documentation for `CountVectorizer` is [here](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html). Familiarize yourself with all settings; try out different combinations and come up with a final model; or rather - two models :).

Our job here is to first map the textual sentiment labels ("Positive"/"Negative") to numeric targets (1 and 0). Then, we build two bag-of-words representations - one for titles and one for review bodies - using CountVectorizer with different hyperparameters. Titles are short signals; reviews are longer and richer. Treating them separately lets us tailor how we extract features from each. Let's do it!

In [36]:
# Binary label - positive is 1, negative is 0
y = reviews['Sentiment'].map({'Positive': 1, 'Negative': 0})

# Vectorizers for title and review 
title_vec = CountVectorizer(
    tokenizer=lambda t: t.split(),   
    lowercase=False,              
    ngram_range=(1, 2),              
    min_df=2,                       
    max_df=0.85,                    
    binary=True,                     
    stop_words=None,
    token_pattern=None
)

review_vec = CountVectorizer(
    tokenizer=lambda t: t.split(),
    lowercase=False,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7,
    max_features=5000,
    binary=False,
    token_pattern=None
)

Let's check out each hyperparameter and talk about its value. Starting with title vectorizer: `tokenizer=lambda t: t.split()` - we will use simple whitespace splitting so the vectorizer sees the same normalized tokens we created manually; keeps tokenization predictable and light. `lowercase=False` - the text was already lowercased in preprocessing, so we turn off internal lowercasing to avoid redundant work. `ngram_range=(1, 2)` - chosen so as to include unigrams and bigrams. Unigrams capture individual sentiment words ("great"), bigrams capture short phrases or negations ("not good"), which are often important in sentiment. `min_df=2` - keeps terms that appear in at least 2 titles; titles are short, so we use a low threshold to avoid throwing away too much signal while still filtering out one-off typos. `max_df=0.85` -
we drop tokens that appear in more than 85% of titles; extremely common terms are unlikely to help distinguish sentiment. `binary=True` - titles are brief, so presence/absence is more meaningful than raw frequency (a word rarely repeats in a title). `stop_words=None` - we do not remove stopwords from titles - because titles are short, dropping common words can erase useful signal. For the review vectorizer: `tokenizer=lambda t: t.split()` - same consistent whitespace-based splitting as for titles. `lowercase=False` - reviews were lowercased earlier; avoid double processing. `stop_words='english'`- remove common English filler words (like "the", "and", "it") that don't carry sentiment, reducing noise. `ngram_range=(1, 2)` - we capture both single words and short phrases/negations; "not good" is distinct from "good". `min_df=5` - we will require a term to appear in at least 5 reviews to be considered, filtering out rare/noisy tokens in the longer text. `max_df=0.7` - we will exclude overly frequent terms (in more than 70% of reviews) that are too generic to help discriminate sentiment. `max_features=5000`- cap the vocabulary size to the top 5000 terms by frequency to keep dimensionality manageable and reduce overfitting. `binary=False` - use raw counts (or frequency) because in reviews, repetition carries strength ("very good good" or multiple occurrences of a positive word reinforce sentiment).

In [37]:
X_title = title_vec.fit_transform(reviews['CleanTitle'])
X_review = review_vec.fit_transform(reviews['CleanReview'])

In [38]:
# Split data once so title and review models are evaluated on the same holdout examples
X_title_train, X_title_test, X_review_train, X_review_test, y_train, y_test = train_test_split(
    X_title, X_review, y, test_size=0.2, stratify=y, random_state=42
)

In [39]:
# Train the 2 models
clf_title = LogisticRegression(solver='liblinear', class_weight='balanced', max_iter=1000)
clf_review = LogisticRegression(solver='liblinear', class_weight='balanced', max_iter=1000)

clf_title.fit(X_title_train, y_train)
clf_review.fit(X_review_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, solver='liblinear')

In [40]:
# Predict
p_title = clf_title.predict_proba(X_title_test)[:, 1]
p_review = clf_review.predict_proba(X_review_test)[:, 1]
y_title_pred = clf_title.predict(X_title_test)
y_review_pred = clf_review.predict(X_review_test)

In [41]:
def evaluate(name, y_true, y_pred, y_proba):
    """
    Function to evaluate model output and construct a report
    """
    
    print(f"\n=== {name} ===")
    print("Classification report:")
    print(classification_report(y_true, y_pred, digits=4))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(f"ROC AUC: {roc_auc_score(y_true, y_proba):.4f}")
    print(f"F1-score: {f1_score(y_true, y_pred):.4f}")

In [42]:
evaluate("Title-only model", y_test, y_title_pred, p_title)
evaluate("Review-only model", y_test, y_review_pred, p_review)


=== Title-only model ===
Classification report:
              precision    recall  f1-score   support

           0     0.7288    0.8113    0.7679        53
           1     0.9587    0.9355    0.9469       248

    accuracy                         0.9136       301
   macro avg     0.8437    0.8734    0.8574       301
weighted avg     0.9182    0.9136    0.9154       301

Confusion matrix:
[[ 43  10]
 [ 16 232]]
ROC AUC: 0.9587
F1-score: 0.9469

=== Review-only model ===
Classification report:
              precision    recall  f1-score   support

           0     0.8679    0.8679    0.8679        53
           1     0.9718    0.9718    0.9718       248

    accuracy                         0.9535       301
   macro avg     0.9198    0.9198    0.9198       301
weighted avg     0.9535    0.9535    0.9535       301

Confusion matrix:
[[ 46   7]
 [  7 241]]
ROC AUC: 0.9868
F1-score: 0.9718


Negative class (0) recall = 0.8113: Of the actual negative reviews, 81% were correctly identified; 19% were missed (false negatives). Positive class (1) recall = 0.9355: Most positives are caught. Precision for negative = 0.7288: When it predicts "negative" - it's right about 73% of the time (some false alarms). ROC AUC = 0.9587: The model ranks positives vs negatives very well overall (high separability). Confusion matrix: 43 true negatives, 10 false positives, 16 false negatives, 232 true positives. So the weakness is mostly in predicting negatives correctly (some are confused as positives).

In [43]:
best = {'weight_title': None, 'f1': -1, 'pred': None, 'proba': None}

for w_title in np.linspace(0, 1, 11):
    w_review = 1 - w_title
    p_comb = w_title * p_title + w_review * p_review
    y_comb = (p_comb >= 0.5).astype(int)
    f1 = f1_score(y_test, y_comb)
    if f1 > best['f1']:
        best.update({'weight_title': w_title, 'f1': f1, 'pred': y_comb, 'proba': p_comb})

print(f"Best weighted ensemble: title weight={best['weight_title']:.2f}")
print(classification_report(y_test, best['pred'], digits=4))
print("ROC AUC:", roc_auc_score(y_test, best['proba']))

Best weighted ensemble: title weight=0.40
              precision    recall  f1-score   support

           0     0.9231    0.9057    0.9143        53
           1     0.9799    0.9839    0.9819       248

    accuracy                         0.9701       301
   macro avg     0.9515    0.9448    0.9481       301
weighted avg     0.9699    0.9701    0.9700       301

ROC AUC: 0.9937614120511261


In conclusion, we trained separate bag-of-words logistic classifiers on titles and review bodies with tailored hyperparameters. The review-only model was stronger, but combining it with the title model via a weighted average (title weight 0.4) improved performance further: the ensemble achieved ROC AUC 0.9938 and notably increased negative-class precision and recall.

### Problem 7. Deep sentiment analysis models (1 point)
Find a suitable model for sentiment analysis in English. Without modifying, training, or fine-tuning the model, make it predict all contents (or better, combinations of titles and contents, if you can). Meaure the accuracy of the model compared to the `sentiment` column in the dataset.

We will use the HuggingFace pretrained model distilbert-base-uncased-finetuned-sst-2-english, exposed via the transformers pipeline for "sentiment-analysis". It's a distilled BERT model fine-tuned on SST-2 (binary sentiment) in English, widely used as a baseline. No training/fine-tuning is required; we just feed it text and it returns "POSITIVE" or "NEGATIVE" with confidence scores. It is suitable for restaurant reviews even if domain-shifted (movie -> restaurant), so we can evaluate how well a generic deep sentiment model works out of the box.

For sentiment prediction without any additional training, we used the pretrained deep model distilbert-base-uncased-finetuned-sst-2-english via the HuggingFace transformers sentiment-analysis pipeline. This distilled BERT model was originally fine-tuned on SST-2 (binary sentiment) and is applied here out of the box (no modification, fine-tuning, or retraining) to three input variants: the review title alone, the review body alone, and the concatenation of title and body. For each variant we collect the model's positive/negative predictions and compared them to the ground-truth sentiment labels.

In [44]:
reviews['TitleInput'] = reviews['Review Title'].fillna("").astype(str)
reviews['ReviewInput'] = reviews['Review'].fillna("").astype(str)
reviews['CombinedInput'] = (reviews['Review Title'].fillna("").astype(str) + ". " +
                            reviews['Review'].fillna("").astype(str))

In [45]:
reviews['TrueLabelBin'] = reviews['Sentiment'].map({'Positive': 1, 'Negative': 0})

In [46]:
sentiment_pipeline = pipeline("sentiment-analysis", 
                              model="distilbert-base-uncased-finetuned-sst-2-english")

Device set to use cpu


In [47]:
def run_and_evaluate(text_col, name):
    """
    Function to receive column and name and predict labels
    """
    
    inputs = reviews[text_col].tolist()
    # Batch inference; truncation ensures long texts are clipped to model max length
    results = sentiment_pipeline(inputs, truncation=True)

    # Extract predicted labels and map to binary
    pred_labels = [r['label'] for r in results]  # 'POSITIVE' / 'NEGATIVE'
    pred_probs = [r['score'] for r in results]   # model confidence (for predicted class)

    reviews[f'Pred_{name}_Label'] = pred_labels
    reviews[f'Pred_{name}_Binary'] = [1 if l == 'POSITIVE' else 0 for l in pred_labels]

    y_true = reviews['TrueLabelBin']
    y_pred = reviews[f'Pred_{name}_Binary']

    print(f"\n=== Evaluation on {name} ===")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Classification report:")
    print(classification_report(y_true, y_pred, digits=4))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

In [48]:
run_and_evaluate('TitleInput', 'Title')
run_and_evaluate('ReviewInput', 'Review')
run_and_evaluate('CombinedInput', 'Combined')


=== Evaluation on Title ===
Accuracy: 0.9374167776298269
Classification report:
              precision    recall  f1-score   support

           0     0.8301    0.8113    0.8206       265
           1     0.9598    0.9644    0.9621      1237

    accuracy                         0.9374      1502
   macro avg     0.8949    0.8879    0.8914      1502
weighted avg     0.9369    0.9374    0.9371      1502

Confusion matrix:
[[ 215   50]
 [  44 1193]]

=== Evaluation on Review ===
Accuracy: 0.9693741677762983
Classification report:
              precision    recall  f1-score   support

           0     0.9132    0.9132    0.9132       265
           1     0.9814    0.9814    0.9814      1237

    accuracy                         0.9694      1502
   macro avg     0.9473    0.9473    0.9473      1502
weighted avg     0.9694    0.9694    0.9694      1502

Confusion matrix:
[[ 242   23]
 [  23 1214]]

=== Evaluation on Combined ===
Accuracy: 0.974034620505992
Classification report:
          

The model achieves 93.7% accuracy using only the title. Positive sentiment is detected very reliably (precision 0.9598, recall 0.9644, F1 0.9621), while negative sentiment is harder (precision 0.8301, recall 0.8113, F1 0.8206). This shows titles carry a strong but somewhat noisier signal - short headlines often condense sentiment but occasionally lack detail, leading to more false positives/negatives for the negative class (50 false positives and 44 false negatives).

Using the full review text yields a substantial gain: 96.9% accuracy. Both classes are predicted with high fidelity (negative F1 0.9132, positive F1 0.9814), and the confusion matrix shows far fewer errors (only 23 false negatives and 23 false positives), reflecting that the longer review provides richer context and disambiguates borderline cases better than the title alone.

Concatenating title and review gives the best result, with 97.4% accuracy. The combined input improves the negative class F1 to 0.9246 (precision 0.9484, recall 0.9019) and further boosts the positive class F1 to 0.9843, while reducing false negatives for positives to 13. This indicates that the title adds complementary signal to the review - helping correct some errors the review-only model made - especially improving precision on negatives and reducing missed positives.

### Problem 8. Deep features (embeddings) (1 point)
Use the same model to perform feature extraction on the review contents (or contents + titles) instead of direct predictions. You should already be familiar how to do that from your work on images.

Use the cosine similarity between texts to try to cluster them. Are there "similar" reviews (you'll need to find a way to measure similarity) across different restaurants? Are customers generally in agreement for the same restaurant?

In [49]:
# Load model/tokenizer 
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()
if torch.cuda.is_available():
    model = model.to("cuda")

In [50]:
def get_embeddings(texts, batch_size=32, max_length=256):
    """
    Function to extract embeddings from a given text
    """
    
    all_emb = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(
                list(batch),
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=max_length
            )
            
            input_ids = enc["input_ids"]
            attention_mask = enc["attention_mask"]
            
            if torch.cuda.is_available():
                input_ids = input_ids.to("cuda")
                attention_mask = attention_mask.to("cuda")
                
            # Get hidden states from DistilBERT backbone
            outputs = model.distilbert(input_ids=input_ids, attention_mask=attention_mask)
            
            last_hidden = outputs.last_hidden_state  # (B, L, H)
            mask = attention_mask.unsqueeze(-1)  # (B, L, 1)
            masked = last_hidden * mask  # zero out padding
            summed = masked.sum(dim=1)  # (B, H)
            lengths = mask.sum(dim=1).clamp(min=1)  # (B, 1)
            mean_pooled = summed / lengths  # (B, H)
            all_emb.append(mean_pooled.cpu().numpy())
    return np.vstack(all_emb)  # (N, hidden_size)

In [51]:
embeddings = get_embeddings(reviews['CombinedInput'])

In [52]:
# Similarity matrix
cos_sim = cosine_similarity(embeddings)  # (N,N)

In [53]:
# Similar reviews across different restaurants
cross_similar = []

for i in range(len(reviews)):
    own_rest = reviews.iloc[i]['Restaurant Name']
    sims = cos_sim[i]
    # mask out self and same restaurant
    candidates = [
        j for j in range(len(reviews))
        if j != i and reviews.iloc[j]['Restaurant Name'] != own_rest
    ]
    
    if not candidates:
        continue
        
    best_j = max(candidates, key=lambda j: sims[j])
    
    cross_similar.append({
        'idx': i,
        'restaurant': own_rest,
        'matched_idx': best_j,
        'matched_restaurant': reviews.iloc[best_j]['Restaurant Name'],
        'similarity': sims[best_j],
        'title_a': reviews.iloc[i]['Review Title'],
        'body_a': reviews.iloc[i]['Review'],
        'title_b': reviews.iloc[best_j]['Review Title'],
        'body_b': reviews.iloc[best_j]['Review'],
    })

In [54]:
# Sort top cross-restaurant similar pairs
top_cross = sorted(cross_similar, key=lambda x: -x['similarity'])[:10]
print("Top cross-restaurant similar review pairs (title+body):")
for rec in top_cross:
    print(f"\n[{rec['similarity']:.3f}] {rec['restaurant']} vs {rec['matched_restaurant']}")
    print(" Title A:", rec['title_a'])
    print(" Body A:", rec['body_a'][:150].replace("\n"," "), "...")
    print(" Title B:", rec['title_b'])
    print(" Body B:", rec['body_b'][:150].replace("\n"," "), "...")

Top cross-restaurant similar review pairs (title+body):

[1.000] The Frog at Bercy Village vs Ad Hoc Ristorante (Piazza del Popolo)
 Title A: Unfortunately disappointing
 Body A: We reserved a table through the website. This was very comfortable. We received a receipt. The atmorphere of the restaurant is very nice. The waitress ...
 Title B: Unfortunately disappointing
 Body B: We reserved a table through the website. This was very comfortable. We received a receipt. The atmorphere of the restaurant is very nice. The waitress ...

[1.000] The Frog at Bercy Village vs Ad Hoc Ristorante (Piazza del Popolo)
 Title A: Expected the TA rating's and I was disapointed
 Body A: Walked in and didn't get in, so made a reservation, confirmed thru e-mail. Arrived on time and they made us feel unwelcomed likethere was no reservati ...
 Title B: Expected the TA rating's and I was disapointed
 Body B: Walked in and didn't get in, so made a reservation, confirmed thru e-mail. Arrived on time and they m

In [55]:
# Intra-restaurant similarity averages
restaurant_groups = reviews.groupby('Restaurant Name').indices
intra_means = {}
intra_all_vals = []
for rest, idxs in restaurant_groups.items():
    if len(idxs) < 2:
        continue
    sub = cos_sim[np.ix_(idxs, idxs)]
    triu = np.triu_indices_from(sub, k=1)
    vals = sub[triu]
    if len(vals) == 0:
        continue
    intra_means[rest] = vals.mean()
    intra_all_vals.extend(vals.tolist())

In [56]:
# Sample inter-restaurant similarities
inter_vals = []
rng = np.random.default_rng(42)
all_idxs = np.arange(len(reviews))
while len(inter_vals) < len(intra_all_vals):
    i, j = rng.choice(all_idxs, size=2, replace=False)
    if reviews.iloc[i]['Restaurant Name'] != reviews.iloc[j]['Restaurant Name']:
        inter_vals.append(cos_sim[i, j])
inter_vals = np.array(inter_vals)
intra_all_vals = np.array(intra_all_vals)

print("\nAverage intra-restaurant similarity (top 10 highest):")
for rest, val in sorted(intra_means.items(), key=lambda x: -x[1])[:10]:
    print(f"  {rest}: {val:.3f}")
print(f"Overall mean intra-restaurant similarity: {intra_all_vals.mean():.3f}")
print(f"Sampled inter-restaurant similarity mean: {inter_vals.mean():.3f}")


Average intra-restaurant similarity (top 10 highest):
  Mosaic: 0.963
  The LOFT: 0.904
  Stara Kamienica: 0.741
  Pelmenya: 0.694
  Ad Hoc Ristorante (Piazza del Popolo): 0.665
  Old Square (Plaza Vieja): 0.600
  The Frog at Bercy Village: 0.499
Overall mean intra-restaurant similarity: 0.598
Sampled inter-restaurant similarity mean: 0.636


In [57]:
# Statistical test: are intra > inter?
t_stat, p_val = stats.ttest_ind(intra_all_vals, inter_vals, equal_var=False)
print(f"\nT-test intra vs inter similarity: t={t_stat:.2f}, p={p_val:.2e}")


T-test intra vs inter similarity: t=-26.86, p=7.91e-159


In [58]:
# Clustering to find themes 
clust = AgglomerativeClustering(n_clusters=None, distance_threshold=0.3, metric="cosine", linkage="average")
cluster_labels = clust.fit_predict(embeddings)
reviews['Cluster'] = cluster_labels
print("\nCluster size (top 10):")
print(reviews['Cluster'].value_counts().head(10))


Cluster size (top 10):
Cluster
1    1234
3     193
0      70
2       4
4       1
Name: count, dtype: int64


Embeddings from the pretrained DistilBERT model capture semantic similarity: many thematically similar reviews appear across different restaurants, indicating common patterns (for example - praise for service or complaints about wait time). Within each restaurant, the average pairwise cosine similarity is significantly higher than between restaurants, showing customers tend to agree about their experience at a given venue. Clustering the embedding space further reveals recurring sentiment/issue themes that span multiple restaurants.

### \* Problem 9. Explore and model at will
In this lab, we focused on preprocessing and feature extraction and we didn't really have a chance to train (or compare) models. The dataset is maybe too small to be conclusive, but feel free to play around with ready-made models, and train your own.

In [59]:
X_title = reviews['TitleInput']
X_review = reviews['ReviewInput']
X_combined = reviews['CombinedInput']
y = reviews['TrueLabelBin']

In [60]:
# Use same split for all variants
X_title_train, X_title_test, y_train, y_test = train_test_split(
    X_title, y, test_size=0.2, stratify=y, random_state=42
)
X_review_train, X_review_test, _, _ = train_test_split(
    X_review, y, test_size=0.2, stratify=y, random_state=42
)
X_combined_train, X_combined_test, _, _ = train_test_split(
    X_combined, y, test_size=0.2, stratify=y, random_state=42
)

In [61]:
def evaluate_pipeline(name, pipeline, X_tr, y_tr, X_te, y_te):
    """
    Helper function to build pipelines and evaluate
    """
    
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(y_te, y_pred))
    print("Classification report:")
    print(classification_report(y_te, y_pred, digits=4))
    print("Confusion matrix:")
    print(confusion_matrix(y_te, y_pred))

In [62]:
def make_tfidf(ngram=(1,2), stop_words='english', min_df=3, max_df=0.85):
    """
    Base TF-IDF vectorizer shared settings (can be customized per variant if desired)
    """
    
    return TfidfVectorizer(ngram_range=ngram, stop_words=stop_words, min_df=min_df, max_df=max_df)

In [63]:
# 1. Logistic Regression
logreg_title = Pipeline([('tfidf', make_tfidf()), ('clf', LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000))])
logreg_review = Pipeline([('tfidf', make_tfidf()), ('clf', LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000))])
logreg_combined = Pipeline([('tfidf', make_tfidf()), ('clf', LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000))])

# 2. Linear SVM with probability calibration
svm_base = LinearSVC(class_weight='balanced', max_iter=5000)
svm_title = Pipeline([('tfidf', make_tfidf()), ('clf', CalibratedClassifierCV(svm_base, cv=3))])
svm_review = Pipeline([('tfidf', make_tfidf()), ('clf', CalibratedClassifierCV(clone(svm_base), cv=3))])
svm_combined = Pipeline([('tfidf', make_tfidf()), ('clf', CalibratedClassifierCV(clone(svm_base), cv=3))])

# 3. Random Forest (on TF-IDF)
rf_title = Pipeline([('tfidf', make_tfidf()), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))])
rf_review = Pipeline([('tfidf', make_tfidf()), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))])
rf_combined = Pipeline([('tfidf', make_tfidf()), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))])

In [64]:
# Title-only
evaluate_pipeline("Logistic Regression (Title)", logreg_title, X_title_train, y_train, X_title_test, y_test)
evaluate_pipeline("SVM (Title)", svm_title, X_title_train, y_train, X_title_test, y_test)
evaluate_pipeline("Random Forest (Title)", rf_title, X_title_train, y_train, X_title_test, y_test)

# Review-only
evaluate_pipeline("Logistic Regression (Review)", logreg_review, X_review_train, y_train, X_review_test, y_test)
evaluate_pipeline("SVM (Review)", svm_review, X_review_train, y_train, X_review_test, y_test)
evaluate_pipeline("Random Forest (Review)", rf_review, X_review_train, y_train, X_review_test, y_test)

# Combined
evaluate_pipeline("Logistic Regression (Combined)", logreg_combined, X_combined_train, y_train, X_combined_test, y_test)
evaluate_pipeline("SVM (Combined)", svm_combined, X_combined_train, y_train, X_combined_test, y_test)
evaluate_pipeline("Random Forest (Combined)", rf_combined, X_combined_train, y_train, X_combined_test, y_test)


=== Logistic Regression (Title) ===
Accuracy: 0.8737541528239202
Classification report:
              precision    recall  f1-score   support

           0     0.6316    0.6792    0.6545        53
           1     0.9303    0.9153    0.9228       248

    accuracy                         0.8738       301
   macro avg     0.7810    0.7973    0.7887       301
weighted avg     0.8777    0.8738    0.8755       301

Confusion matrix:
[[ 36  17]
 [ 21 227]]

=== SVM (Title) ===
Accuracy: 0.9036544850498339
Classification report:
              precision    recall  f1-score   support

           0     0.7857    0.6226    0.6947        53
           1     0.9228    0.9637    0.9428       248

    accuracy                         0.9037       301
   macro avg     0.8542    0.7932    0.8188       301
weighted avg     0.8986    0.9037    0.8991       301

Confusion matrix:
[[ 33  20]
 [  9 239]]

=== Random Forest (Title) ===
Accuracy: 0.867109634551495
Classification report:
              precis

In [65]:
def get_oof_probs(pipeline, X_full, y_full, splits=5):
    """
    Get out-of-fold probabilities on training set for stacking
    """
    
    skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=42)
    oof = np.zeros(len(X_full))
    
    for train_idx, val_idx in skf.split(X_full, y_full):
        model = clone(pipeline)
        model.fit(X_full.iloc[train_idx], y_full.iloc[train_idx])
        try:
            oof[val_idx] = model.predict_proba(X_full.iloc[val_idx])[:,1]
        except:
            # fallback to decision function via sigmoid
            df = model.decision_function(X_full.iloc[val_idx])
            oof[val_idx] = 1 / (1 + np.exp(-df))
    return oof

In [66]:
# Build meta-features on training set (combined text)
X_meta_train = X_combined_train.reset_index(drop=True)
y_meta_train = y_train.reset_index(drop=True)
oof_logreg = get_oof_probs(logreg_combined, X_meta_train, y_meta_train)
oof_rf = get_oof_probs(rf_combined, X_meta_train, y_meta_train)
meta_train = np.vstack([oof_logreg, oof_rf]).T

In [67]:
# Train meta-model
meta_clf = LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000)
meta_clf.fit(meta_train, y_meta_train)

LogisticRegression(class_weight='balanced', max_iter=1000, solver='liblinear')

In [68]:
def get_base_probs(pipeline, X):
    """
    Build test-level meta-features
    """
    
    try:
        return pipeline.predict_proba(X)[:,1]
    except:
        df = pipeline.decision_function(X)
        return 1 / (1 + np.exp(-df))

In [69]:
p_logreg_test = get_base_probs(logreg_combined, X_combined_test)
p_rf_test = get_base_probs(rf_combined, X_combined_test)
meta_test = np.vstack([p_logreg_test, p_rf_test]).T
p_stacked = meta_clf.predict_proba(meta_test)[:,1]
y_stacked = (p_stacked >= 0.5).astype(int)

print("\n=== Stacked Ensemble (Logistic + RF on Combined) ===")
print("Accuracy:", accuracy_score(y_test, y_stacked))
print(classification_report(y_test, y_stacked, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_stacked))


=== Stacked Ensemble (Logistic + RF on Combined) ===
Accuracy: 0.9601328903654485
              precision    recall  f1-score   support

           0     0.8361    0.9623    0.8947        53
           1     0.9917    0.9597    0.9754       248

    accuracy                         0.9601       301
   macro avg     0.9139    0.9610    0.9351       301
weighted avg     0.9643    0.9601    0.9612       301

Confusion matrix:
[[ 51   2]
 [ 10 238]]


We compared multiple classical sentiment classification pipelines without using deep embeddings. Logistic regression, linear SVM, and random forest were trained on TF-IDF features from titles, reviews, and combined text. The combined title+review input consistently outperformed individual components, and a simple stacking ensemble of logistic regression and random forest further improved robustness by blending their strengths. This confirms that combining lexical signals and model diversity yields better generalization on this modest dataset.